In [1]:
from pyspark.sql import functions as F

In [2]:
from spark_session import get_spark

In [3]:
import sources

In [4]:
spark = get_spark("joins")

In [5]:
orders = sources.orders(spark)
products = sources.products(spark)
customers = sources.customers(spark)
print("orders: ", orders.count())

orders:  282000


In [6]:
orders.show(10, truncate=False)

+--------+-------------------+-----------+----------+--------+-------+-------------------+
|order_id|order_ts           |customer_id|product_id|quantity|status |updated_at         |
+--------+-------------------+-----------+----------+--------+-------+-------------------+
|260001  |2026-10-14 04:57:11|1124       |P002      |1       |paid   |2026-10-14 04:57:11|
|260002  |2026-10-14 04:38:27|150        |P023      |3       |paid   |2026-10-14 04:38:27|
|260003  |2026-10-14 06:03:59|834        |P016      |2       |shipped|2026-10-14 06:03:59|
|260004  |2026-10-14 12:54:48|190        |P011      |1       |shipped|2026-10-14 12:54:48|
|260005  |2026-10-14 18:51:44|1223       |P023      |1       |paid   |2026-10-14 18:51:44|
|260006  |2026-10-14 15:52:46|880        |P023      |2       |paid   |2026-10-14 15:52:46|
|260007  |2026-10-14 15:35:37|292        |P020      |1       |paid   |2026-10-14 15:35:37|
|260008  |2026-10-14 19:07:06|1377       |P014      |1       |paid   |2026-10-14 19:07:06|

In [7]:
inner = orders.join(products, on="product_id")
left = orders.join(products, on="product_id", how="left")
anti = orders.join(products, on="product_id", how="left_anti")

print("inner: ", inner.count(), "left: ", left.count(), "anti: ", anti.count())
anti.groupBy("product_id").count().show()

inner:  275053 left:  282000 anti:  6947
+----------+-----+
|product_id|count|
+----------+-----+
|      P041| 6947|
+----------+-----+



In [8]:
with_cust = orders.join(customers, on="customer_id")
print("orders with a customer: ", orders.where("customer_id IS NOT NULL").count(), "joined: ", with_cust.count())
dupes = customers.groupBy("customer_id").count().where("count > 1")
print("dupes customer id: ", dupes.count())

fixed = orders.join(customers.dropDuplicates(["customer_id"]), "customer_id")
print("after dedup: ", fixed.count())

orders with a customer:  276355 joined:  279191
dupes customer id:  20
after dedup:  276355


In [9]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
orders.join(products, "product_id").explain()
orders.join (F.broadcast(products), "product_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#3, order_id#0L, order_ts#1, customer_id#2, quantity#4, status#5, updated_at#6, name#8, category#9, price#10]
   +- SortMergeJoin [product_id#3], [product_id#7], Inner
      :- Sort [product_id#3 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(product_id#3, 8), ENSURE_REQUIREMENTS, [plan_id=1029]
      :     +- Filter isnotnull(product_id#3)
      :        +- FileScan csv [order_id#0L,order_ts#1,customer_id#2,product_id#3,quantity#4,status#5,updated_at#6] Batched: false, DataFilters: [isnotnull(product_id#3)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/d:/Learn PySpark/data/raw/orders], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:bigint,order_ts:timestamp,customer_id:int,product_id:string,quantity:int,status:s...
      +- Sort [product_id#7 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(product_id#7, 8), ENSURE_REQUI

In [ ]:
spark.stop()